# Entity Memory

> **Extract named entities from every turn, maintain a structured knowledge base of people, places, and preferences, and inject relevant entity facts into each prompt. This gives the agent persistent, structured understanding of the user's world.**

Imagine a personal assistant who keeps a contact card for every person, place, and project you mention. Each time you share new info ("Sarah got promoted"), they update the right card. Next time you ask about Sarah, they pull her card and give you a complete answer. Entity Memory works the same way.

Previous memory techniques store conversation verbatim (buffer), trim by recency (sliding window), compress into prose (summary), or retrieve by semantic similarity (vector store). None of them *understand* what was said at a structured level.

**Entity Memory** takes a different approach. After each turn, the system extracts entities (people, places, preferences, projects) and stores structured facts about each one. When the user says "My manager Sarah approved the Berlin trip," the system creates or updates records for *Sarah* (role: manager) and *Berlin* (context: upcoming trip). On later turns, when the user mentions Sarah or Berlin, the relevant facts are retrieved and injected into the prompt.

**The payoff:** an agent that doesn't merely *recall* that something was said. It *knows* structured facts about the entities in the user's life. It can answer questions like "What do you know about Sarah?" without scanning raw history.

**By the end of this notebook you'll understand:**
- How to build an entity extractor using Claude's tool-use (function calling) API.
- A key-value entity store that accumulates facts per entity.
- How entity context is injected into prompts for informed responses.
- Cross-conversation persistence: saving and reloading entity knowledge.
- How entity memory compares to sliding window on structured recall tasks.

## Key Concepts

- **Entity extraction:** Using an LLM (via tool-use / function calling) to identify entities mentioned in each turn: people, places, organizations, preferences, projects. This is more flexible than rule-based NER (Named Entity Recognition, the task of spotting and classifying proper nouns).
- **Entity store:** A key-value data structure mapping entity names to structured records (type, facts list, metadata). Think of it as the agent's "address book" about the user's world.
- **Fact accumulation:** Each time an entity appears with new information, the facts list grows. You can detect and resolve duplicates and contradictions.
- **Entity-context injection:** Before each LLM call, entities mentioned in the current message are looked up. Their facts are included in the system prompt as structured context.
- **Cross-conversation persistence:** The entity store serializes to JSON and reloads in a new session. This gives the agent memory that spans conversations.
- **Disambiguation:** Handling cases where the same name refers to different entities (e.g., two people named "Alex") or different names refer to the same entity (e.g., "mom" and "Linda").

## Architecture

<p align="center">
  <img src="../../images/diagrams/07_entity_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph Turn["Each Conversation Turn"]
        A["User message"] --> B["Entity Extractor\n(Claude tool-use)"]
        B --> C["Extracted entities\n+ new facts"]
        C --> D["Update\nEntity Store"]
    end

    subgraph Store["Entity Store (key-value)"]
        D --> E[("{ name -> {\n  type, facts[],\n  first_seen,\n  last_seen\n} }")]
    end

    subgraph Response["Response Generation"]
        A --> F["Lookup mentioned\nentities"]
        E --> F
        F --> G["Build prompt:\nsystem + entity context\n+ recent messages"]
        G --> H["LLM\n(Claude)"]
        H --> I["Response"]
    end

    subgraph Persist["Persistence"]
        E --> J["Save to JSON"]
        J --> K["Load in\nnew session"]
        K --> E
    end

    style E fill:#4f46e5,color:#fff
    style H fill:#059669,color:#fff
    style B fill:#d97706,color:#fff
```

</details>

In [ ]:
# Install required packages (run once)
%pip install -q anthropic python-dotenv matplotlib numpy

Load environment variables and initialize the Anthropic client. You need an `ANTHROPIC_API_KEY` in your `.env` file.

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()  # reads API keys from .env

import anthropic

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"

print("\u2713 API key loaded")
print(f"\u2713 anthropic version: {anthropic.__version__}")

## Core Implementation

Our Entity Memory system has three components:

1. **`EntityStore`:** A dictionary-based store mapping entity names to structured records (type, facts, timestamps). Supports save/load to JSON for cross-session persistence.
2. **`EntityExtractor`:** Uses Claude's tool-use API to extract entities and facts from each message. The LLM acts as a structured NER system via function calling.
3. **`EntityMemory`:** The main class that orchestrates extraction, storage, context injection, and chat.

In [ ]:
class EntityStore:
    """Key-value store mapping entity names to structured records."""

    def __init__(self):
        self.entities: dict[str, dict] = {}

    def update_entity(self, name: str, entity_type: str, facts: list[str]) -> None:
        """Create or update an entity with new facts."""
        key = name.lower().strip()
        now = datetime.now().isoformat()

        if key not in self.entities:
            self.entities[key] = {
                "name": name,
                "type": entity_type,
                "facts": [],
                "first_seen": now,
                "last_seen": now,
            }

        record = self.entities[key]
        record["last_seen"] = now
        # Only add genuinely new facts (simple dedup via lowercase comparison)
        existing_lower = {f.lower() for f in record["facts"]}
        for fact in facts:
            if fact.lower() not in existing_lower:
                record["facts"].append(fact)
                existing_lower.add(fact.lower())

    def get_entity(self, name: str) -> dict | None:
        """Retrieve an entity record by name."""
        return self.entities.get(name.lower().strip())

    def find_mentioned(self, text: str) -> list[dict]:
        """Find all stored entities mentioned in a text string."""
        text_lower = text.lower()
        found = []
        for key, record in self.entities.items():
            if key in text_lower:
                found.append(record)
        return found

    def summary(self) -> str:
        """Return a formatted summary of all stored entities."""
        if not self.entities:
            return "No entities stored."
        lines = []
        for key, record in self.entities.items():
            facts_str = "; ".join(record["facts"])
            lines.append(f"  {record['name']} ({record['type']}): {facts_str}")
        return "\n".join(lines)



We add persistence methods to `EntityStore`. The `save` method writes all entities to a JSON file. The `load` class method reads them back. This lets the agent's knowledge survive between sessions.

In [ ]:
    def save(self, path: str) -> None:
        """Persist entity store to a JSON file."""
        with open(path, "w") as f:
            json.dump(self.entities, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "EntityStore":
        """Load entity store from a JSON file."""
        store = cls()
        with open(path) as f:
            store.entities = json.load(f)
        return store

    def __len__(self) -> int:
        return len(self.entities)

    def __repr__(self) -> str:
        return f"EntityStore({len(self.entities)} entities)"


print("\u2713 EntityStore class defined")

We define the tool schema (a JSON description) that tells Claude how to call our entity extraction function. Tool-use (also called function calling) lets the LLM output structured data instead of free text. The schema specifies what fields to extract: entity name, type, and a list of facts.

In [ ]:
# Define the tool schema for entity extraction via Claude function calling
EXTRACT_ENTITIES_TOOL = {
    "name": "store_entities",
    "description": (
        "Extract and store named entities mentioned in the conversation. "
        "Call this tool with ALL entities (people, places, organizations, "
        "preferences, projects, pets, etc.) and their associated facts."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "entities": {
                "type": "array",
                "description": "List of entities extracted from the message.",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The entity name (e.g., 'Sarah', 'Berlin', 'Project Atlas').",
                        },
                        "entity_type": {
                            "type": "string",
                            "enum": ["person", "place", "organization", "preference",
                                     "project", "pet", "event", "other"],
                            "description": "The type/category of this entity.",
                        },
                        "facts": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "List of facts learned about this entity from the message.",
                        },
                    },
                    "required": ["name", "entity_type", "facts"],
                },
            },
        },
        "required": ["entities"],
    },
}




The `EntityExtractor` wraps a Claude API call that uses the tool schema above. It sends the conversation exchange to Claude and asks it to extract all entities. Claude returns structured JSON through the tool-use interface.

In [ ]:
class EntityExtractor:
    """Uses Claude tool-use to extract entities and facts from messages."""

    def __init__(self, model: str = "claude-sonnet-4-20250514"):
        self.client = anthropic.Anthropic()
        self.model = model

    def extract(self, user_msg: str, assistant_msg: str) -> list[dict]:
        """Extract entities from a user-assistant exchange.

        Returns a list of dicts with keys: name, entity_type, facts.
        """
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=(
                "You are an entity extraction system. Analyze the conversation "
                "and extract ALL named entities (people, places, organizations, "
                "preferences, projects, pets, events, etc.) along with any facts "
                "learned about them. Always call the store_entities tool. "
                "If no entities are found, call it with an empty list."
            ),
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Extract entities from this exchange:\n\n"
                        f"User: {user_msg}\n"
                        f"Assistant: {assistant_msg}"
                    ),
                }
            ],
            tools=[EXTRACT_ENTITIES_TOOL],
            tool_choice={"type": "tool", "name": "store_entities"},
        )

        # Parse the tool call result
        for block in response.content:
            if block.type == "tool_use" and block.name == "store_entities":
                return block.input.get("entities", [])
        return []


print("\u2713 EntityExtractor class defined (uses Claude tool-use)")

The `EntityMemory` class orchestrates everything. It holds an `EntityStore` and an `EntityExtractor`. When building the system prompt, it looks up entities mentioned in the user's current message and injects their facts as context.

In [ ]:
class EntityMemory:
    """Chat agent with entity memory: extract, store, and inject entity context."""

    def __init__(
        self,
        chat_model: str = "claude-sonnet-4-20250514",
        extraction_model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
        recent_buffer_size: int = 10,
        entity_store: EntityStore | None = None,
    ):
        self.client = anthropic.Anthropic()
        self.chat_model = chat_model
        self.base_system_prompt = system_prompt or "You are a helpful assistant."
        self.max_tokens = max_tokens
        self.recent_buffer_size = recent_buffer_size

        # Components
        self.store = entity_store or EntityStore()
        self.extractor = EntityExtractor(model=extraction_model)

        # State
        self.messages: list[dict] = []
        self.turn_count = 0

    def _build_system_prompt(self, user_input: str) -> str:
        """Inject relevant entity context into the system prompt."""
        parts = [self.base_system_prompt]

        # Find entities mentioned in the current message
        mentioned = self.store.find_mentioned(user_input)
        if mentioned:
            entity_context = "\n".join(
                f"- {r['name']} ({r['type']}): {'; '.join(r['facts'])}"
                for r in mentioned
            )
            parts.append(
                f"You have the following knowledge about entities mentioned "
                f"by the user:\n\n{entity_context}\n\n"
                f"Use this knowledge to give informed, personalized responses."
            )

        # Also provide a brief overview of all known entities
        if self.store.entities:
            all_names = ", ".join(
                r["name"] for r in self.store.entities.values()
            )
            parts.append(f"All known entities: {all_names}")

        return "\n\n".join(parts)



The `chat` method is the main loop. It builds the prompt with entity context, calls Claude for a response, then runs entity extraction on the exchange. Any new entities or facts get stored for future turns.

In [ ]:
    def chat(self, user_input: str) -> str:
        """Send a message, extract entities, and respond with entity context."""
        self.turn_count += 1

        # Build prompt with entity context
        system = self._build_system_prompt(user_input)

        # Add user message to recent buffer
        self.messages.append({"role": "user", "content": user_input})
        recent = self.messages[-self.recent_buffer_size:]

        # Generate response
        response = self.client.messages.create(
            model=self.chat_model,
            max_tokens=self.max_tokens,
            system=system,
            messages=recent,
        )
        assistant_text = response.content[0].text

        # Store assistant reply in buffer
        self.messages.append({"role": "assistant", "content": assistant_text})

        # Extract entities from this exchange
        try:
            extracted = self.extractor.extract(user_input, assistant_text)
            for entity in extracted:
                self.store.update_entity(
                    name=entity["name"],
                    entity_type=entity["entity_type"],
                    facts=entity["facts"],
                )
        except Exception as e:
            print(f"  [extraction warning: {e}]")

        return assistant_text



We add utility methods for inspecting the entity store, saving to disk, and loading from a saved file. The `from_saved` class method creates a pre-loaded `EntityMemory` from a JSON file, which is how you resume across sessions.

In [ ]:
    def known_entities(self) -> str:
        """Return a formatted summary of all known entities."""
        return self.store.summary()

    def save_memory(self, path: str) -> None:
        """Persist entity store to disk."""
        self.store.save(path)

    @classmethod
    def from_saved(cls, path: str, **kwargs) -> "EntityMemory":
        """Create an EntityMemory instance pre-loaded with saved entities."""
        store = EntityStore.load(path)
        return cls(entity_store=store, **kwargs)

    def __repr__(self) -> str:
        return (
            f"EntityMemory(turns={self.turn_count}, "
            f"entities={len(self.store)})"
        )


print("\u2713 EntityMemory class defined")

## Usage Example: Entity Extraction in Action

Let's have a multi-turn conversation that mentions several people, places, and preferences. After each turn, the system extracts entities and accumulates structured facts. Later turns benefit from this knowledge.

In [ ]:
mem = EntityMemory(
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)

messages = [
    "Hi! My name is Priya and I'm a data scientist at Spotify.",
    "My partner Alex is a chef -- he runs a Thai restaurant called Basil & Lime in Portland.",
    "We have a rescue greyhound named Zephyr who loves long walks.",
    "I'm working on a project called Wavelength -- it's a music recommendation engine.",
    "Alex and I are planning a trip to Tokyo next spring.",
    "What do you know about Alex?",
    "What project am I working on?",
]

for msg in messages:
    print(f"\U0001f464 User:  {msg}")
    reply = mem.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print(f"\n\U0001f4ca Entities tracked: {len(mem.store)}")

Let's inspect every entity the system extracted. For each entity, you'll see its type, the accumulated facts, and when it was first and last seen. This is the agent's structured "address book" about the user's world.

In [ ]:
# Inspect the entity store
print("=== Entity Store Contents ===\n")
for key, record in mem.store.entities.items():
    print(f"\U0001f4cc {record['name']} ({record['type']})")
    for fact in record["facts"]:
        print(f"   \u2022 {fact}")
    print(f"   First seen: {record['first_seen'][:19]}")
    print(f"   Last seen:  {record['last_seen'][:19]}")
    print()

## Cross-Conversation Persistence

A key advantage of entity memory: it's easy to serialize. The entity store is a dictionary. Save it to JSON, load it in a new session, and the agent picks up right where it left off.

This simulates what happens when a user closes a chat, then opens a new conversation the next day.

In [ ]:
# -- Save entity store from Session 1 --
ENTITY_FILE = "entity_store.json"
mem.save_memory(ENTITY_FILE)
print(f"\u2713 Saved {len(mem.store)} entities to {ENTITY_FILE}")

# -- Simulate a new session (Session 2) --
print("\n--- NEW SESSION ---\n")

mem2 = EntityMemory.from_saved(
    ENTITY_FILE,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)
print(f"Loaded entities: {len(mem2.store)}")
print(f"Known entities: {[r['name'] for r in mem2.store.entities.values()]}\n")

# The new session should know about Priya, Alex, Zephyr, etc.
session2_questions = [
    "What's the name of my dog?",
    "Where does Alex work?",
    "Tell me about the Wavelength project.",
]

for msg in session2_questions:
    print(f"\U0001f464 User:  {msg}")
    reply = mem2.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print("\u2713 Cross-conversation persistence works -- no re-introduction needed!")

## Experiment: Entity Memory vs. Sliding Window

We'll run a controlled test:

1. Plant **10 entity facts** across a **30-turn conversation** (mixed with filler).
2. Ask **10 recall questions** about specific entities.
3. Compare **Entity Memory** (extracts & stores structured facts) vs. **Sliding Window** (keeps last 10 messages).

The hypothesis: entity memory will recall structured facts from any turn. The sliding window will forget facts once they leave the window.

In [ ]:
# Facts planted at specific turns
PLANTED_FACTS = {
    1:  ("My name is Jordan Rivera and I'm 34 years old.",     "jordan",    "name/age"),
    3:  ("My best friend Mika teaches physics at MIT.",         "mika",      "friend"),
    5:  ("I have a cat named Pixel -- she's a black tabby.",    "pixel",     "pet"),
    8:  ("My company is called NovaSpark -- we build AR glasses.", "novaspark", "company"),
    11: ("My mom Linda lives in Tucson and she's a retired nurse.", "linda",  "mother"),
    15: ("I'm training for a triathlon in Lake Tahoe this August.", "tahoe",  "event"),
    18: ("My partner Sam is a violinist in the Chicago Symphony.", "sam",     "partner"),
    22: ("I just bought a cabin near Bend, Oregon.",            "bend",      "property"),
    26: ("My therapist Dr. Patel recommended I start journaling.", "patel",  "therapist"),
    29: ("I'm reading 'Project Hail Mary' by Andy Weir right now.", "hail mary", "book"),
}

FILLER_MESSAGES = [
    "What's the difference between HTTP and HTTPS?",
    "Can you explain quantum entanglement simply?",
    "What's a good recipe for banana bread?",
    "How does a heat pump work?",
    "Tell me about the history of jazz.",
    "What's the tallest mountain in South America?",
    "How do noise-canceling headphones work?",
    "What causes auroras?",
    "Explain the Pythagorean theorem.",
    "What's the oldest known civilization?",
    "How does sourdough starter work?",
    "What is the Doppler effect?",
    "Tell me about coral reefs.",
    "How does memory work in the brain?",
    "What's the speed of light?",
    "How do electric cars regenerate braking energy?",
    "What is dark energy?",
    "Tell me about the Silk Road trade routes.",
    "How do birds navigate during migration?",
    "What is the Krebs cycle?",
]



We build the 30-turn conversation by interleaving planted facts with filler. Then we define recall questions. Each question targets a specific entity fact and includes a keyword we'll check in the answer.

In [ ]:
# Build 30-turn conversation
conversation_30 = []
filler_idx = 0
for turn in range(1, 31):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_30.append((turn, msg))

RECALL_QUESTIONS = [
    ("What is my full name?",                    "jordan",      1),
    ("Who is Mika?",                             "physics",     3),
    ("What pet do I have?",                      "pixel",       5),
    ("What's my company called?",                "novaspark",   8),
    ("Tell me about my mom.",                    "linda",       11),
    ("What sporting event am I training for?",   "triathlon",   15),
    ("Who is Sam?",                              "violinist",   18),
    ("Do I own any property?",                   "bend",        22),
    ("Who is Dr. Patel?",                        "therapist",   26),
    ("What book am I reading?",                  "hail mary",   29),
]

print(f"Conversation: {len(conversation_30)} turns")
print(f"Facts planted at turns: {sorted(PLANTED_FACTS.keys())}")
print(f"Recall questions: {len(RECALL_QUESTIONS)}")

We run all 30 turns through entity memory. After each turn, the extractor identifies entities and stores facts. Watch the entity count grow as the conversation progresses.

In [ ]:
# -- Run 30 turns through Entity Memory --

ent_mem = EntityMemory(
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Entity Memory...")
for turn_num, msg in conversation_30:
    ent_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done ({len(ent_mem.store)} entities tracked)")

print(f"\n\u2713 Done. Entities tracked: {len(ent_mem.store)}")
print(f"\n=== Extracted Entities ===")
print(ent_mem.known_entities())

We define a sliding window baseline and run the same 30 turns through it. The window keeps only the last 10 messages, so older facts will drop out of context.

In [ ]:
class SlidingWindowBaseline:
    """Sliding window: keeps only the last K messages."""

    def __init__(self, window_size: int = 10, model: str = "claude-sonnet-4-20250514",
                 system_prompt: str | None = None, max_tokens: int = 1024):
        self.window_size = window_size
        self.client = anthropic.Anthropic()
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        window = self.messages[-self.window_size:]

        kwargs = dict(model=self.model, max_tokens=self.max_tokens, messages=window)
        if self.system_prompt:
            kwargs["system"] = self.system_prompt

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


# -- Run 30 turns through Sliding Window --

sw_mem = SlidingWindowBaseline(
    window_size=10,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_30:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done")

print(f"\n\u2713 Done. Window keeps last {sw_mem.window_size} messages.")

Now we run the recall test on both systems. For each question, we check whether the answer contains the expected keyword. Entity memory should recall facts from any turn because it stores them permanently. The sliding window will lose facts that have scrolled past.

In [ ]:
# -- Recall test: Entity Memory --
print("=== Entity Memory - Recall Test ===\n")
ent_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = ent_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    ent_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

# -- Recall test: Sliding Window --
print("=== Sliding Window - Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

ent_score = sum(1 for r in ent_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Entity Memory score:       {ent_score}/{len(RECALL_QUESTIONS)}")
print(f"Sliding Window score:      {sw_score}/{len(RECALL_QUESTIONS)}")

Let's visualize the comparison. Panel 1 shows total recall scores. Panel 2 breaks down recall by fact age: which planted facts did each system remember?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Panel 1: Total recall scores --
ent_score = sum(1 for r in ent_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
labels = ["Entity\nMemory", "Sliding Window\n(K=10)"]
scores = [ent_score, sw_score]
colors = ["#4f46e5", "#ef4444"]

bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title(f"Total Facts Recalled (out of {len(RECALL_QUESTIONS)})")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- Panel 2: Recall by fact age --
fact_turns = [r["planted_at"] for r in ent_results]
ent_recalled = [1 if r["recalled"] else 0 for r in ent_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, ent_recalled, width, label="Entity Memory", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=9)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()

plt.tight_layout()
plt.savefig("entity_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nEntity Memory:   {ent_score}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window:  {sw_score}/{len(RECALL_QUESTIONS)} recalled")

## Discussion & Tradeoffs

### Strengths
- **Structured knowledge.** Unlike raw buffer or summary, entity memory stores *facts about things*, not conversation fragments. This enables precise recall of attributes (names, roles, preferences).
- **Cross-session persistence.** The entity store serializes to JSON in one line. A new session loads prior knowledge without replaying history.
- **Composable.** Entity memory works well alongside other techniques. Combine it with a sliding window for recent context and entities for long-term structured knowledge.
- **Interpretable.** You can inspect exactly what the agent "knows" about each entity. It's a transparent, auditable knowledge base.

### Weaknesses
- **Extraction cost.** Each turn requires an extra LLM call for entity extraction (via tool-use). This roughly doubles the per-turn API cost.
- **Extraction imperfection.** LLM-based extraction can miss entities, hallucinate facts, or extract spurious entities from filler conversation.
- **Disambiguation.** Handling two entities with the same name, or aliases for the same entity ("mom" vs. "Linda"), requires additional logic not covered here.
- **Schema rigidity.** Our entity types are fixed. Real-world entities may not fit neatly into categories. More flexible approaches use free-text fact storage.
- **Fact staleness.** Facts are only added, never expired. Contradictions (e.g., "I moved to Berlin" after previously storing "lives in Portland") need explicit conflict resolution.

### When to Use Entity Memory

| Scenario | Recommendation |
|----------|---------------|
| Personal assistants that track user info across sessions | Excellent. The core use case. |
| CRM-style agents that manage contacts and relationships | Excellent. |
| Short conversations with no returning users | Overkill. Sliding window is sufficient. |
| High-volume, cost-sensitive deployments | Careful: the extraction call doubles cost. |
| Need to recall exact conversation flow/phrasing | Use buffer or vector store instead. |
| Combining with other memory types | Great complement to sliding window + summary. |

### Cost Model
Per turn:
- **1 LLM call** for response generation (same as any approach)
- **1 LLM call** for entity extraction via tool-use (~50-200 tokens output)
- **Entity store lookup** is free (in-memory dictionary)

The extraction call is the main added cost. For production, consider extracting entities asynchronously or batching extraction every *N* turns.

## Further Reading

- [Anthropic Tool Use (Function Calling)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): The API used for entity extraction in this notebook
- [LangChain ConversationEntityMemory](https://python.langchain.com/docs/modules/memory/types/entity_summary_memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Framework-level entity memory implementation
- [Mem0: Self-Improving Memory Layer](https://github.com/mem0ai/mem0): Production memory system with entity tracking
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior" (2023)](https://arxiv.org/abs/2304.03442): Entity memory in simulated agents
- [spaCy Named Entity Recognition](https://spacy.io/usage/linguistic-features?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques#named-entities): Traditional NER as an alternative to LLM extraction
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Multi-turn conversation patterns

---

*← Previous: [06 - Vector Store Memory](../06_vector_store_memory/) · Next: [08 - Knowledge Graph Memory](../08_knowledge_graph_memory/) →*

In [ ]:
# Clean up temp files
import os
for f in ["entity_store.json", "entity_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Entity aliases
Add an `aliases` field to `EntityStore` so that 'Bob', 'Robert', and 'Bob Smith' all resolve to the same entity record. Update `find_mentioned()` to check aliases. Test with a conversation that uses multiple names for the same person.

### Challenge 2: Extraction precision and recall
Prepare 10 conversation turns with known entities and facts as ground truth. Run `EntityExtractor.extract()` on each turn and count true positives, false positives, and false negatives. Compute precision and recall. Identify which entity types the extractor misses most often.

### Challenge 3: Entity relationship tracking
Extend `EntityStore` to record relationships between entities (e.g., 'Alice works-at Acme Corp'). Store them as a list of (subject, predicate, object) triples per entity. This bridges directly to the approach in 08 Knowledge Graph Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--07-entity-memory--entity-memory)
